# Assignment 11

## Data mirroring
We begin by doubling the size of the dataset by mirroring both the cut and uncut files around the y-axis.

In [1]:
import pandas as pd
import os
import glob


folders = [
    "../../data/kinect_good_preprocessed_not_cut_A11_mediapipe",
    "../../data/kinect_good_preprocessed_A9_mediapipe"
]

pairs = [
    ('left_shoulder', 'right_shoulder'),
    ('left_elbow', 'right_elbow'),
    ('left_wrist', 'right_wrist'),
    ('left_hip', 'right_hip'),
    ('left_knee', 'right_knee'),
    ('left_ankle', 'right_ankle')
]

for target_folder in folders:
    all_files = glob.glob(os.path.join(target_folder, "*.csv"))
    print(f"Processing folder: {target_folder} ({len(all_files)} files found)")
    
    count = 0
    for f in all_files:
        if "_mirrored" in f: 
            continue
        
        df = pd.read_csv(f)
        mirrored_df = df.copy()
        
        x_cols = [c for c in df.columns if "_3d_x" in c]
        mirrored_df[x_cols] = 1.0 - df[x_cols]
        
        for left, right in pairs:
            for axis in ['x', 'y', 'z']:
                l_col = f"{left}_3d_{axis}"
                r_col = f"{right}_3d_{axis}"
                
                if l_col in df.columns and r_col in df.columns:
                    mirrored_df[l_col] = df[r_col]
                    mirrored_df[r_col] = df[l_col]
                    
                    if axis == 'x':
                        mirrored_df[l_col] = 1.0 - mirrored_df[l_col]
                        mirrored_df[r_col] = 1.0 - mirrored_df[r_col]

        new_path = f.replace(".csv", "_mirrored.csv")
        mirrored_df.to_csv(new_path, index=False)
        count += 1

    print(f"Successfully created {count} mirrored files in {target_folder}.\n")

print("All folders processed.")

Processing folder: ../../data/kinect_good_preprocessed_not_cut_A11_mediapipe (360 files found)
Successfully created 180 mirrored files in ../../data/kinect_good_preprocessed_not_cut_A11_mediapipe.

Processing folder: ../../data/kinect_good_preprocessed_A9_mediapipe (358 files found)
Successfully created 179 mirrored files in ../../data/kinect_good_preprocessed_A9_mediapipe.

All folders processed.


## Data labelling - Creating data
Here we join the uncut and cut files to create the actual data. This creates the files we use for training, validation aswell as testing.

In [2]:
import pandas as pd
import os
import glob
from tqdm import tqdm


cut_folder = "../../data/kinect_good_preprocessed_A9_mediapipe"
uncut_folder = "../../data/kinect_good_preprocessed_not_cut_A11_mediapipe"
output_folder = "../../data/labeled_files" 

os.makedirs(output_folder, exist_ok=True)

uncut_files = glob.glob(os.path.join(uncut_folder, "*.csv"))

print(f"Found {len(uncut_files)} uncut files. Starting labeling...")

for uncut_path in tqdm(uncut_files):
    filename = os.path.basename(uncut_path)
    cut_path = os.path.join(cut_folder, filename)
    
    if not os.path.exists(cut_path):
        print(f"Warning: No cut file found for {filename}. Skipping.")
        continue
        
    df_uncut = pd.read_csv(uncut_path)
    df_cut = pd.read_csv(cut_path)
    

    start_frame = df_cut['FrameNo'].min()
    end_frame = df_cut['FrameNo'].max()
    
    # Label 1 if the frame is within the range, otherwise 0
    df_uncut['activity_label'] = (
        (df_uncut['FrameNo'] >= start_frame) & 
        (df_uncut['FrameNo'] <= end_frame)
    ).astype(int)
    
    save_path = os.path.join(output_folder, filename)
    df_uncut.to_csv(save_path, index=False)

print(f"\nDone! Labeled files saved to: {output_folder}")

Found 360 uncut files. Starting labeling...


 37%|███▋      | 132/360 [00:01<00:02, 112.17it/s]

 98%|█████████▊| 352/360 [00:03<00:00, 120.50it/s]

100%|██████████| 360/360 [00:03<00:00, 112.72it/s]


Done! Labeled files saved to: ../../data/labeled_files


## Training the model

## Configuration

In [ ]:
config = {
    "epochs": 30,
    "patience": 5,
    "batch_size": 64,
    "lr": 0.0005,
    "hidden_size": 128,
    "num_layers": 2,
    "seq_length": 5,  # Set to 5 for CPU, 30 for GPU (Optimal)
    "dropout": 0.2,
    "input_size": 39   # 13 joints * 3 (x,y,z)
}

### Classes

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset

class ActivityGatekeeper(nn.Module):
    def __init__(self, config):
        super(ActivityGatekeeper, self).__init__()
        self.lstm = nn.LSTM(
            config["input_size"], 
            config["hidden_size"], 
            config["num_layers"], 
            batch_first=True, 
            dropout=config["dropout"]
        )
        self.fc = nn.Linear(config["hidden_size"], 1) 

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class ActivityDataset(Dataset):
    def __init__(self, file_paths, config):
        self.X, self.y = [], []
        seq_length = config["seq_length"]
        for path in file_paths:
            df = pd.read_csv(path)
            feat_cols = [c for c in df.columns if any(s in c for s in ['_x','_y','_z']) and 'FrameNo' not in c]
            X_data, y_data = df[feat_cols].values.astype('float32'), df['activity_label'].values.astype('float32')
            if len(df) < seq_length: continue
            for i in range(len(df) - seq_length + 1):
                self.X.append(X_data[i : i + seq_length])
                self.y.append(y_data[i + seq_length - 1])
    def __len__(self): return len(self.y)
    def __getitem__(self, idx): return torch.tensor(self.X[idx]), torch.tensor([self.y[idx]])

def get_with_mirrors(file_list):
    res = []
    for f in file_list:
        res.append(f)
        m = f.replace(".csv", "_mirrored.csv")
        if os.path.exists(m): res.append(m)
    return res

### Resample to make the dataset more balanced

In [5]:
import numpy as np

data_path = "../../data/labeled_files/"
originals = sorted([f for f in glob.glob(os.path.join(data_path, "*.csv")) if "_mirrored" not in f])

all_labels = []
for f in originals:
    all_labels.extend(pd.read_csv(f)['activity_label'].values)

neg, pos = np.bincount(np.array(all_labels).astype(int))
pos_weight = torch.tensor([neg / pos], dtype=torch.float).to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
print(f"Imbalance ratio: {neg/pos:.2f}")

Imbalance ratio: 0.57


### Training and evaluation

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.utils.data import DataLoader
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

kf = KFold(n_splits=10, shuffle=True, random_state=42)
fold_metrics = []

best_overall_f1 = 0.0
best_fold_info = ""

for fold, (train_idx, test_idx) in enumerate(kf.split(originals)):
    print(f"\n--- Fold {fold+1} ---")
    
    # Splots files instead of segments
    train_orig = [originals[i] for i in train_idx]
    test_orig = [originals[i] for i in test_idx]
    
    v_split = int(len(train_orig) * 0.9)
    # Adding mirrors only to training/val
    train_files = get_with_mirrors(train_orig[:v_split])
    val_files = get_with_mirrors(train_orig[v_split:])
    test_files = test_orig 

    # Creating loaders contating the data
    train_loader = DataLoader(ActivityDataset(train_files, config), batch_size=config["batch_size"], shuffle=True)
    val_loader = DataLoader(ActivityDataset(val_files, config), batch_size=config["batch_size"], shuffle=False)
    test_loader = DataLoader(ActivityDataset(test_files, config), batch_size=config["batch_size"], shuffle=False)

    model = ActivityGatekeeper(config).to(device)

    optimizer = optim.Adam(model.parameters(), lr=config["lr"]) 
    # This resamples the data
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    
    # Initializing values
    best_v_loss, patience_count = float('inf'), 0
    fold_checkpoint_path = f"best_fold_{fold}.pth"


    # Each fold does max epochs 
    for epoch in range(config["epochs"]):
        model.train()
        for x, y in train_loader:
            optimizer.zero_grad()
            criterion(model(x.to(device)), y.to(device)).backward()
            optimizer.step()
        
        model.eval()
        v_loss = 0
        with torch.no_grad():
            for x, y in val_loader: 
                v_loss += criterion(model(x.to(device)), y.to(device)).item()
        
        avg_v = v_loss / len(val_loader)
        if avg_v < best_v_loss:
            best_v_loss = avg_v
            patience_count = 0
            torch.save(model.state_dict(), fold_checkpoint_path)
        else:
            patience_count += 1
            if patience_count >= config["patience"]: break


    model.load_state_dict(torch.load(fold_checkpoint_path)) 
    model.eval()
    t_true, t_preds = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = torch.sigmoid(model(inputs))
            t_preds.extend((outputs > 0.5).float().cpu().numpy())
            t_true.extend(labels.numpy())

    # Evaluation 
    p = precision_score(t_true, t_preds, zero_division=0)
    r = recall_score(t_true, t_preds, zero_division=0)
    f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0
    
    print(f"Fold {fold+1} Results -> Precision: {p:.4f}, Recall: {r:.4f}, F1: {f1:.4f}")

    # Only saves best model
    if f1 > best_overall_f1:
        best_overall_f1 = f1
        torch.save(model.state_dict(), "gatekeeper_best.pth")
        best_fold_info = f"Fold {fold+1} (P: {p:.4f}, R: {r:.4f}, F1: {f1:.4f})"
        print(f"🌟 New Absolute Best Model found in Fold {fold+1}!")

print("\n" + "="*30)
print(f"TRAINING COMPLETE")
print(f"Best Model preserved from: {best_fold_info}")
print("Final model saved as: gatekeeper_best.pth")
print("="*30)

Using device: cpu

--- Fold 1 ---


# Creating confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(t_true, t_preds)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', xticklabels=['Idle', 'Active'], yticklabels=['Idle', 'Active'])
plt.title("Confusion Matrix @ 0.50 Threshold")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.show()

# One file check for manual control

In [ ]:
import torch
import pandas as pd
import numpy as np

def extract_exercise_bounds(file_path, model_path, config):
    # 1. Setup Device and Model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ActivityGatekeeper(config).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # 2. Load and Prepare Data
    df = pd.read_csv(file_path)
    feat_cols = [c for c in df.columns if any(s in c for s in ['_x','_y','_z']) and 'FrameNo' not in c]
    X_data = df[feat_cols].values.astype('float32')
    frame_numbers = df['FrameNo'].values
    
    seq_length = config["seq_length"]
    predictions = []

    # 3. Sliding Window Inference
    with torch.no_grad():
        for i in range(len(df) - seq_length + 1):
            # Extract window
            window = X_data[i : i + seq_length]
            window_tensor = torch.tensor(window).unsqueeze(0).to(device) # Add batch dimension
            
            # Predict
            output = model(window_tensor)
            prob = torch.sigmoid(output).item()
            predictions.append(1 if prob >= 0.5 else 0)

    # 4. Find Start and Stop
    # We pad the beginning because the first (seq_length - 1) frames don't have enough context
    full_preds = [0] * (seq_length - 1) + predictions
    
    start_frame = None
    stop_frame = None

    for i in range(1, len(full_preds)):
        # Start: 0 -> 1 transition
        if full_preds[i-1] == 0 and full_preds[i] == 1:
            if start_frame is None: start_frame = frame_numbers[i]
        
        # Stop: 1 -> 0 transition
        if full_preds[i-1] == 1 and full_preds[i] == 0:
            stop_frame = frame_numbers[i-1]

    return start_frame, stop_frame

# --- HOW TO USE ---
test_csv = uncut_path + "/A1_kinect.csv" 
model_file = "gatekeeper_best.pth"

start, stop = extract_exercise_bounds(test_csv, model_file, config)

print(f"File: {test_csv}")
print(f"Detected Start Frame: {start}")
print(f"Detected Stop Frame: {stop}")
if start and stop:
    print(f"Total Exercise Duration: {stop - start} frames")
else:
    print("Warning: Model did not detect a clear start/stop transition.")